In [1]:
import pandas as pd
from PIL import Image
import os
from tqdm import tqdm

from transformers import AutoModel, AutoTokenizer
import torch

In [2]:
model_name = 'minicpm'
prompt_type = 'zeroshot'

In [3]:
dataset = 'type1'
image_type = 'simple'
device = "cuda:0"

In [ ]:
qas = pd.read_json(f'../{dataset}/qa.json')
qas

In [5]:
image_indices = qas['image_index'].values.astype(int)
questions = qas['question'].values
answers = qas['answer'].values

In [6]:
image_base_path = f'../{dataset}/{image_type}/'

In [ ]:
all_images = os.listdir(image_base_path)
index_to_image = {}

if dataset == 'type1':
    prefix = 'multichart_'
    if image_type == 'original' or image_type == 'simple':
        sep = '_'
    else:
        sep = '.'

for index in set(image_indices):
    for image in all_images:
        if image.startswith(f'{prefix}{index}{sep}'):
            if index not in index_to_image:
                index_to_image[index] = []
            index_to_image[index].append(image)

index_to_image

In [8]:
message_template = [
    {
        "role": "user",
        "content": []
    }
]

In [9]:
def get_message(image_index, prompt, question):
    images = []
    for image in index_to_image[image_index]:
        image_path = f'{image_base_path}{image}'
        images.append(Image.open(image_path).convert('RGB'))
    message = message_template.copy()
    message[0]['content'] = images
    message[0]['content'].append(prompt.format(question))
    return message

In [ ]:
model = AutoModel.from_pretrained('openbmb/MiniCPM-V-2_6',                                                    
    trust_remote_code=True,
    attn_implementation='flash_attention_2', 
    torch_dtype=torch.bfloat16,
    device_map = device,
    cache_dir='drive/multichartqa/models_cache') # sdpa or flash_attention_2, no eager

model = model.eval()
tokenizer = AutoTokenizer.from_pretrained('openbmb/MiniCPM-V-2_6', trust_remote_code=True)

In [11]:
model_responses = []

In [12]:
def run_model():
    for i, question in enumerate(tqdm(questions)):
        image_index = image_indices[i]
        prompt = """Your task is the answer the question based on the given image. Your final answer to the question should strictly be in the format - "Final Answer:" <final_answer>.\n\nQuestion: {}""" 
        messages = get_message(image_index, prompt, question)
        answer = model.chat(image=None, msgs=messages, tokenizer=tokenizer)
        model_responses.append({'question_id': i, 'question': question, 'gold': answers[i], 'response': answer.strip()})
        # print(f'Question: {question}\nAnswer: {answers[i]}\nResponse: {answer.strip()}\n\n')

In [ ]:
run_model()

In [14]:
# saving the model responses
os.makedirs(f'../model_responses/{dataset}', exist_ok=True)

model_responses_df = pd.DataFrame(model_responses)
model_responses_df.to_json(f'../model_responses/{dataset}/{model_name}_{image_type}_{prompt_type}.json', orient='records')